In [107]:
import numpy as np 
import pandas as pd 
import os
import joblib
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv
/kaggle/input/datasets/vedikagupta0/data-work-students-health-risk/preprocessed_students-health-risk-pred-dataset.csv
/kaggle/input/datasets/vedikagupta0/data-work-students-health-risk/preprocessing_artifacts.pkl


In [108]:
artifacts = joblib.load("preprocessing_artifacts.pkl")

imputer = artifacts["imputer"]
target_maps = artifacts["target_maps"]
mappings = artifacts["mappings"]

In [109]:
def preprocessor_fe(df, imputer, target_maps, mappings):
    int_cols = ['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','water_intake','exercise_duration']
    str_cols=['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level',
           'smoking_alcohol', 'gender']
    
    for col in int_cols:
        df[col] = df[col].fillna(imputer[col])
        
    df[str_cols] = df[str_cols].fillna(-1)
    
    for col in str_cols:
    
        tmp = df[col].astype(str).str.lower()
    
        df[f"{col}_te"] = tmp.map(target_maps[col]).fillna(-1)
    
        df[col] = tmp.map(mappings[col]).fillna(-1)
        
    df['exercise_or_not'] = df['exercise_duration']>0
    df['is_normal_heartrate'] = (df['heart_rate']>60) & (df['heart_rate']<100)
    
    for col in str_cols:
        df[f"{col}_missing_ind"] = df[col]==-1
        
    for col1, col2 in combinations(int_cols, 2):
        df[f"{col1}_by_{col2}"] = df[col1] / df[col2].replace(0, np.nan)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(-1, inplace=True)
    return df

In [110]:
df = preprocessor_fe(pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv'),imputer, target_maps,mappings)

In [112]:
df.isnull().sum()

id                                          0
health_condition                            0
sleep_duration                              0
heart_rate                                  0
bmi                                         0
calorie_expenditure                         0
step_count                                  0
exercise_duration                           0
water_intake                                0
diet_type                                   0
stress_level                                0
sleep_quality                               0
physical_activity_level                     0
smoking_alcohol                             0
gender                                      0
diet_type_te                                0
stress_level_te                             0
sleep_quality_te                            0
physical_activity_level_te                  0
smoking_alcohol_te                          0
gender_te                                   0
exercise_or_not                   

In [113]:
X = df.drop(["health_condition", "id"], axis=1)
y = df["health_condition"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [116]:
logreg = LogisticRegression(
    max_iter=500,
    random_state=42,
    multi_class="multinomial"
)

logreg.fit(X_train_scaled, y_train)

y_pred = logreg.predict(X_test_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [117]:
coef_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "coefficient": logreg.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df.head(50))

                                     feature  coefficient
14                           stress_level_te     7.841040
22                  stress_level_missing_ind     6.894424
16                physical_activity_level_te     4.472097
24       physical_activity_level_missing_ind     4.263284
30              sleep_duration_by_step_count     0.952651
21                     diet_type_missing_ind     0.861612
13                              diet_type_te     0.847529
8                               stress_level    -0.579766
35                  heart_rate_by_step_count    -0.463483
26                        gender_missing_ind     0.421817
18                                 gender_te     0.418537
42         calorie_expenditure_by_step_count    -0.237520
29     sleep_duration_by_calorie_expenditure     0.231872
39                         bmi_by_step_count    -0.183448
38                bmi_by_calorie_expenditure    -0.139290
47         water_intake_by_exercise_duration    -0.099238
0             

In [118]:
y_pred = logreg.predict(X_test_scaled)

print("Balanced Accuracy:",
      balanced_accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Balanced Accuracy: 0.8416238140444353

Classification Report:
              precision    recall  f1-score   support

     at-risk       0.96      0.98      0.97    118512
         fit       0.87      0.78      0.82      7961
   unhealthy       0.88      0.77      0.82     11545

    accuracy                           0.95    138018
   macro avg       0.91      0.84      0.87    138018
weighted avg       0.95      0.95      0.95    138018


Confusion Matrix:
[[116485    902   1125]
 [  1748   6182     31]
 [  2692     16   8837]]


In [119]:
scores = cross_val_score(
    logreg,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="balanced_accuracy"
)

print("CV Balanced Accuracy:", scores.mean())
print("Fold Scores:", scores)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

CV Balanced Accuracy: 0.8407771982948203
Fold Scores: [0.84095771 0.84060761 0.8425416  0.84079249 0.83898659]


In [120]:
y.value_counts()

health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

In [121]:
test = preprocessor_fe(pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv'),imputer, target_maps,mappings)

In [125]:
test_id = test['id']
test_scaled = pd.DataFrame(
    scaler.transform(test.drop(columns='id')),
    columns=X_test.columns,
    index=test.index
)
test_y_pred = logreg.predict(test_scaled)


In [126]:
submission = pd.DataFrame(
    {'id':test_id,
    'health_condition':test_y_pred}
)

In [128]:
submission.to_csv('submission.csv', index=False)